In [51]:
%run Transformer_Architecture.ipynb

tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])
tensor([[ 9, 10,  5, 23, 13, 13,  0,  0,  0,  0],
        [ 1, 11, 17,  0,  0,  0,  0,  0,  0,  0],
        [ 5,  1, 21,  1,  3, 12,  0,  0,  0,  0],
        [ 1, 15,  5, 10,  1, 17, 15, 19,  0,  0]])
tensor([[[-2.4853, -1.0031,  0.0385,  ..., -2.2465,  0.3161,  1.7152],
         [ 0.7372,  2.5853, -2.5986,  ...,  2.8018,  1.0823, -0.1604],
         [-0.1947,  1.1773, -0.9625,  ...,  2.2777,  2.4015,  0.0722],
         ...,
         [ 0.9035, -0.5262,  2.7525,  ...,  0.2521,  0.1096,  2.3922],
         [ 0.8966, -0.1104,  1.0972,  ...,  0.7954,  1.2197,  3.9095],
         [ 1.5210, -0.1252, -0.1209,  ...,  1.6520,  0.0000,  2.8175]],

        [[ 0.3309, -0.1655,  0.7646,  ..., -0.0000, -0.0292,  1.8643],
         [ 1.7122,  2.5069,  0.8834,  ...,  2.0918,  2.4531,  1.0198],
         [-1.4813, -0.0000, -0.8794,  ...,  0.9072,  0

In [52]:
@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 6
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # MoE config
    use_moe: bool = True
    n_experts: int = 4
    moe_hidden_mult: int = 4
    router_aux_loss_coef: float = 0.01

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [53]:
class ExpertFFN(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        hidden_dim = config.moe_hidden_mult * config.d_model

        self.net = nn.Sequential(
            nn.Linear(config.d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        """
        x: [num_tokens_for_this_expert, d_model]
        """
        return self.net(x)

In [54]:
class Top1MoEFeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config
        self.n_experts = config.n_experts
        self.d_model = config.d_model

        # Router maps each token representation to expert logits.
        self.router = nn.Linear(
            config.d_model,
            config.n_experts,
            bias=False,
        )

        self.experts = nn.ModuleList([
            ExpertFFN(config)
            for _ in range(config.n_experts)
        ])

    def load_balance_loss(self, router_probs, selected_experts):
        """
        router_probs:      [num_tokens, n_experts]
        selected_experts:  [num_tokens]

        Encourages tokens to be spread across experts.
        """

        num_tokens = router_probs.shape[0]

        expert_mask = F.one_hot(
            selected_experts,
            num_classes=self.n_experts,
        ).float()
        # [num_tokens, n_experts]

        tokens_per_expert = expert_mask.mean(dim=0)
        # [n_experts]

        prob_per_expert = router_probs.mean(dim=0)
        # [n_experts]

        aux_loss = self.n_experts * torch.sum(
            tokens_per_expert * prob_per_expert
        )

        return aux_loss

    def forward(self, x):
        """
        x: [batch_size, seq_len, d_model]

        returns:
            output:   [batch_size, seq_len, d_model]
            aux_loss: scalar
            stats:    routing statistics
        """

        batch_size, seq_len, d_model = x.shape

        x_flat = x.reshape(-1, d_model)
        # [B*T, D]

        router_logits = self.router(x_flat)
        # [B*T, n_experts]

        router_probs = torch.softmax(router_logits, dim=-1)
        # [B*T, n_experts]

        top1_prob, top1_expert = torch.max(router_probs, dim=-1)
        # top1_prob:   [B*T]
        # top1_expert: [B*T]

        output_flat = torch.zeros_like(x_flat)
        # [B*T, D]

        for expert_id, expert in enumerate(self.experts):
            token_mask = top1_expert == expert_id

            if token_mask.any():
                expert_input = x_flat[token_mask]
                # [tokens_for_expert, D]

                expert_output = expert(expert_input)
                # [tokens_for_expert, D]

                expert_output = expert_output * top1_prob[token_mask, None]
                # Scale by router confidence

                output_flat[token_mask] = expert_output

        output = output_flat.view(batch_size, seq_len, d_model)
        # [B, T, D]

        aux_loss = self.load_balance_loss(
            router_probs=router_probs,
            selected_experts=top1_expert,
        )

        with torch.no_grad():
            expert_counts = torch.bincount(
                top1_expert,
                minlength=self.n_experts,
            )

        stats = {
            "expert_counts": expert_counts,
            "router_probs_mean": router_probs.mean(dim=0).detach(),
        }

        return output, aux_loss, stats

In [55]:
class MoEDecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = Top1MoEFeedForward(config)

    def forward(self, x, attention_mask=None):
        """
        x: [batch_size, seq_len, d_model]
        """

        # Attention residual
        x = x + self.attn(
            self.ln_1(x),
            attention_mask=attention_mask,
        )

        # MoE FFN residual
        moe_out, moe_aux_loss, moe_stats = self.mlp(self.ln_2(x))

        x = x + moe_out

        if attention_mask is not None:
            x = x * attention_mask[:, :, None].to(x.dtype)

        return x, moe_aux_loss, moe_stats

In [56]:
class MoEGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = TokenEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            MoEDecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # Weight tying
        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:      [B, T]
        attention_mask: [B, T]
        labels:         [B, T]
        """

        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.config.max_seq_len

        x = self.embedding(input_ids)
        # [B, T, D]

        moe_aux_losses = []
        moe_stats_by_layer = []

        for block in self.blocks:
            x, moe_aux_loss, moe_stats = block(
                x,
                attention_mask=attention_mask,
            )

            moe_aux_losses.append(moe_aux_loss)
            moe_stats_by_layer.append(moe_stats)

        x = self.final_ln(x)
        # [B, T, D]

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        lm_loss = None
        moe_aux_loss = None
        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            lm_loss = F.cross_entropy(
                shift_logits.reshape(-1, self.config.vocab_size),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

            moe_aux_loss = torch.stack(moe_aux_losses).mean()

            loss = lm_loss + self.config.router_aux_loss_coef * moe_aux_loss

        return {
            "logits": logits,
            "loss": loss,
            "lm_loss": lm_loss,
            "moe_aux_loss": moe_aux_loss,
            "moe_stats_by_layer": moe_stats_by_layer,
        }

In [57]:
torch.manual_seed(42)

batch_size = 3
max_seq_len = 6
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len),
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(attention_mask == 0, -100)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,

    # MoE config
    use_moe=True,
    n_experts=4,
    moe_hidden_mult=4,
    router_aux_loss_coef=0.01,
)

model = MoEGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(outputs["logits"].shape)

print("\ntotal loss:")
print(outputs["loss"])

print("\nlm loss:")
print(outputs["lm_loss"])

print("\nmoe aux loss:")
print(outputs["moe_aux_loss"])

print("\nExpert routing stats:")
for layer_idx, stats in enumerate(outputs["moe_stats_by_layer"]):
    print(f"\nLayer {layer_idx}")
    print("expert_counts:", stats["expert_counts"])
    print("router_probs_mean:", stats["router_probs_mean"])

outputs["loss"].backward()

print("\nRouter grad exists?")
print(model.blocks[0].mlp.router.weight.grad is not None)

print("\nExpert 0 grad exists?")
print(model.blocks[0].mlp.experts[0].net[0].weight.grad is not None)

lengths:
tensor([1, 6, 5])

input_ids:
tensor([[59,  0,  0,  0,  0,  0],
        [ 3, 68, 77, 81, 82, 91],
        [23, 93, 59, 40, 32,  0]])

attention_mask:
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])

labels:
tensor([[  59, -100, -100, -100, -100, -100],
        [   3,   68,   77,   81,   82,   91],
        [  23,   93,   59,   40,   32, -100]])

logits shape:
torch.Size([3, 6, 100])

total loss:
tensor(26.4312, grad_fn=<AddBackward0>)

lm loss:
tensor(26.4209, grad_fn=<NllLossBackward0>)

moe aux loss:
tensor(1.0359, grad_fn=<MeanBackward0>)

Expert routing stats:

Layer 0
expert_counts: tensor([4, 3, 7, 4])
router_probs_mean: tensor([0.2026, 0.2661, 0.3040, 0.2273])

Layer 1
expert_counts: tensor([9, 2, 3, 4])
router_probs_mean: tensor([0.2778, 0.2371, 0.2374, 0.2478])

Router grad exists?
True

Expert 0 grad exists?
True


In [58]:
n_experts =  4
tokens_per_expert = torch.randint(0, 2, (10, 4))
print(tokens_per_expert)
prob_per_expert = torch.rand(10, 4)
print(prob_per_expert)
print(torch.sum(
            tokens_per_expert * prob_per_expert
        ))
aux_loss = n_experts * torch.sum(
            tokens_per_expert * prob_per_expert
        )
print(aux_loss)



tensor([[0, 1, 0, 0],
        [0, 0, 1, 1],
        [0, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]])
tensor([[0.1444, 0.9828, 0.2015, 0.4556],
        [0.6592, 0.2413, 0.6690, 0.4487],
        [0.7457, 0.5087, 0.9386, 0.7858],
        [0.2668, 0.5100, 0.1403, 0.6795],
        [0.4000, 0.4284, 0.0717, 0.5463],
        [0.1545, 0.7036, 0.8253, 0.2822],
        [0.4571, 0.8890, 0.3687, 0.0639],
        [0.2628, 0.7572, 0.7879, 0.1857],
        [0.3779, 0.0735, 0.2385, 0.1333],
        [0.9133, 0.2916, 0.9820, 0.5894]])
tensor(4.0554)
tensor(16.2217)


In [59]:
# Top 2 MOE aka Sparse MOE Routing (tokens are routed through 2 MOE Experts also know as sparse MoE routing since only some experts are activates)

# Suppose router probabilities for one token are:

# expert 0: 0.10
# expert 1: 0.60
# expert 2: 0.25
# expert 3: 0.05

# Top-2 experts are:
# expert 1 with probability 0.60
# expert 2 with probability 0.25

# expert 1 weight = 0.60 / (0.60 + 0.25) = 0.7059
# expert 2 weight = 0.25 / (0.60 + 0.25) = 0.2941

# output = 0.7059 * expert_1(x) + 0.2941 * expert_2(x)

In [60]:

@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 6
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # MoE config
    use_moe: bool = True
    n_experts: int = 4
    moe_top_k: int = 2
    moe_hidden_mult: int = 4
    router_aux_loss_coef: float = 0.01

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads


In [61]:
class ExpertFFN(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        hidden_dim = config.moe_hidden_mult * config.d_model

        self.net = nn.Sequential(
            nn.Linear(config.d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        """
        x: [num_tokens_for_this_expert, d_model]
        """
        return self.net(x)

In [62]:
class Top2MoEFeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        assert config.moe_top_k == 2, "This implementation is specifically Top-2 MoE"
        assert config.n_experts >= 2, "Need at least 2 experts for Top-2 MoE"

        self.config = config
        self.n_experts = config.n_experts
        self.top_k = config.moe_top_k
        self.d_model = config.d_model

        # Router: maps each token representation to expert logits.
        self.router = nn.Linear(
            config.d_model,
            config.n_experts,
            bias=False,
        )

        self.experts = nn.ModuleList([
            ExpertFFN(config)
            for _ in range(config.n_experts)
        ])

    def load_balance_loss(self, router_probs, topk_experts):
        """
        router_probs:
            [num_valid_tokens, n_experts]

        topk_experts:
            [num_valid_tokens, top_k]

        Encourages balanced expert usage.
        """

        # One-hot selected experts.
        selected_mask = F.one_hot(
            topk_experts,
            num_classes=self.n_experts,
        ).float()
        # [N, top_k, n_experts]

        # Fraction of selected expert slots assigned to each expert.
        # Sums to 1 across experts.
        tokens_per_expert = selected_mask.mean(dim=(0, 1))
        # [n_experts]

        # Average router probability mass per expert.
        prob_per_expert = router_probs.mean(dim=0)
        # [n_experts]

        aux_loss = self.n_experts * torch.sum(
            tokens_per_expert * prob_per_expert
        )

        return aux_loss

    def forward(self, x, attention_mask=None):
        """
        x:
            [batch_size, seq_len, d_model]

        attention_mask:
            [batch_size, seq_len]

        returns:
            output:   [batch_size, seq_len, d_model]
            aux_loss: scalar
            stats:    routing statistics
        """

        batch_size, seq_len, d_model = x.shape

        x_flat = x.reshape(-1, d_model)
        # [B*T, D]

        if attention_mask is None:
            valid_token_mask = torch.ones(
                x_flat.shape[0],
                device=x.device,
                dtype=torch.bool,
            )
        else:
            valid_token_mask = attention_mask.reshape(-1).bool()
            # [B*T]

        x_valid = x_flat[valid_token_mask]
        # [num_valid_tokens, D]

        output_flat = torch.zeros_like(x_flat)
        # [B*T, D]

        if x_valid.numel() == 0:
            aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
            stats = {
                "expert_counts": torch.zeros(
                    self.n_experts,
                    device=x.device,
                    dtype=torch.long,
                ),
                "router_probs_mean": torch.zeros(
                    self.n_experts,
                    device=x.device,
                    dtype=x.dtype,
                ),
            }
            return output_flat.view(batch_size, seq_len, d_model), aux_loss, stats

        router_logits = self.router(x_valid)
        # [N, n_experts]

        router_probs = torch.softmax(router_logits, dim=-1)
        # [N, n_experts]

        topk_probs, topk_experts = torch.topk(
            router_probs,
            k=self.top_k,
            dim=-1,
        )
        # topk_probs:   [N, 2]
        # topk_experts: [N, 2]

        # Normalize top-2 probabilities so they sum to 1.
        topk_probs = topk_probs / topk_probs.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-9)
        # [N, 2]

        output_valid = torch.zeros_like(x_valid)
        # [N, D]

        # Route tokens to selected experts.
        for expert_id, expert in enumerate(self.experts):
            # Find all token/slot pairs where this expert was selected.
            token_idx, expert_slot = (topk_experts == expert_id).nonzero(
                as_tuple=True
            )

            if token_idx.numel() == 0:
                continue

            expert_input = x_valid[token_idx]
            # [num_tokens_for_expert, D]

            expert_output = expert(expert_input)
            # [num_tokens_for_expert, D]

            weights = topk_probs[token_idx, expert_slot].unsqueeze(-1)
            # [num_tokens_for_expert, 1]

            weighted_output = expert_output * weights
            # [num_tokens_for_expert, D]

            # Add contribution back to the corresponding token.
            # A token receives contribution from two experts.  i.e sum of 2 D dimensional (d_model) weighted vectors  
            output_valid.index_add_(
                dim=0,
                index=token_idx,
                source=weighted_output,
            )

        output_flat[valid_token_mask] = output_valid

        output = output_flat.view(batch_size, seq_len, d_model)
        # [B, T, D]

        aux_loss = self.load_balance_loss(
            router_probs=router_probs,
            topk_experts=topk_experts,
        )

        with torch.no_grad():
            expert_counts = torch.bincount(
                topk_experts.reshape(-1),
                minlength=self.n_experts,
            )

            router_probs_mean = router_probs.mean(dim=0)

        stats = {
            "expert_counts": expert_counts,
            "router_probs_mean": router_probs_mean.detach(),
        }

        return output, aux_loss, stats

In [63]:
class Top2MoEDecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = Top2MoEFeedForward(config)

    def forward(self, x, attention_mask=None):
        """
        x: [batch_size, seq_len, d_model]
        """

        # Attention residual
        x = x + self.attn(
            self.ln_1(x),
            attention_mask=attention_mask,
        )

        # Top-2 MoE residual
        moe_out, moe_aux_loss, moe_stats = self.mlp(
            self.ln_2(x),
            attention_mask=attention_mask,
        )

        x = x + moe_out

        if attention_mask is not None:
            x = x * attention_mask[:, :, None].to(x.dtype)

        return x, moe_aux_loss, moe_stats

In [64]:
class Top2MoEGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = TokenEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            Top2MoEDecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # Weight tying
        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:      [B, T]
        attention_mask: [B, T]
        labels:         [B, T]
        """

        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.config.max_seq_len

        x = self.embedding(input_ids)
        # [B, T, D]

        moe_aux_losses = []
        moe_stats_by_layer = []

        for block in self.blocks:
            x, moe_aux_loss, moe_stats = block(
                x,
                attention_mask=attention_mask,
            )

            moe_aux_losses.append(moe_aux_loss)
            moe_stats_by_layer.append(moe_stats)

        x = self.final_ln(x)

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        lm_loss = None
        moe_aux_loss = None
        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            lm_loss = F.cross_entropy(
                shift_logits.reshape(-1, self.config.vocab_size),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

            moe_aux_loss = torch.stack(moe_aux_losses).mean()

            loss = lm_loss + self.config.router_aux_loss_coef * moe_aux_loss

        return {
            "logits": logits,
            "loss": loss,
            "lm_loss": lm_loss,
            "moe_aux_loss": moe_aux_loss,
            "moe_stats_by_layer": moe_stats_by_layer,
        }

In [65]:
torch.manual_seed(42)

batch_size = 3
max_seq_len = 6
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len),
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(attention_mask == 0, -100)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,

    # Top-2 MoE config
    use_moe=True,
    n_experts=4,
    moe_top_k=2,
    moe_hidden_mult=4,
    router_aux_loss_coef=0.01,
)

model = Top2MoEGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(outputs["logits"].shape)

print("\ntotal loss:")
print(outputs["loss"])

print("\nlm loss:")
print(outputs["lm_loss"])

print("\nmoe aux loss:")
print(outputs["moe_aux_loss"])

print("\nExpert routing stats:")
for layer_idx, stats in enumerate(outputs["moe_stats_by_layer"]):
    print(f"\nLayer {layer_idx}")
    print("expert_counts:", stats["expert_counts"])
    print("router_probs_mean:", stats["router_probs_mean"])

outputs["loss"].backward()

print("\nRouter grad exists?")
print(model.blocks[0].mlp.router.weight.grad is not None)

print("\nExpert 0 grad exists?")
print(model.blocks[0].mlp.experts[0].net[0].weight.grad is not None)

print("\nExpert 1 grad exists?")
print(model.blocks[0].mlp.experts[1].net[0].weight.grad is not None)

lengths:
tensor([1, 6, 5])

input_ids:
tensor([[59,  0,  0,  0,  0,  0],
        [ 3, 68, 77, 81, 82, 91],
        [23, 93, 59, 40, 32,  0]])

attention_mask:
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])

labels:
tensor([[  59, -100, -100, -100, -100, -100],
        [   3,   68,   77,   81,   82,   91],
        [  23,   93,   59,   40,   32, -100]])

logits shape:
torch.Size([3, 6, 100])

total loss:
tensor(26.3065, grad_fn=<AddBackward0>)

lm loss:
tensor(26.2965, grad_fn=<NllLossBackward0>)

moe aux loss:
tensor(1.0004, grad_fn=<MeanBackward0>)

Expert routing stats:

Layer 0
expert_counts: tensor([7, 5, 6, 6])
router_probs_mean: tensor([0.2465, 0.2423, 0.2528, 0.2584])

Layer 1
expert_counts: tensor([6, 6, 6, 6])
router_probs_mean: tensor([0.2685, 0.2388, 0.2520, 0.2407])

Router grad exists?
True

Expert 0 grad exists?
True

Expert 1 grad exists?
True


In [66]:
# DeepSeekMoE : Towards Ultimate Expert Specialization in Mixture-of-Experts Language Models. 
# The paper’s two main MoE ideas are: fine-grained expert segmentation and shared expert isolation.
# In other words, instead of only having routed experts, DeepSeekMoE also keeps some shared experts that every token
# always uses, while routed experts are selected sparsely by the router.

In [67]:
@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 6
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # DeepSeekMoE-style config
    n_routed_experts: int = 8
    n_shared_experts: int = 1
    moe_top_k: int = 2
    moe_hidden_mult: int = 4
    router_aux_loss_coef: float = 0.01

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [68]:
class ExpertFFN(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        hidden_dim = config.moe_hidden_mult * config.d_model

        self.net = nn.Sequential(
            nn.Linear(config.d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        """
        x: [num_tokens, d_model]
        """
        return self.net(x)

In [69]:
# DeepSeekMoE-style FeedForward

class DeepSeekMoEFeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        assert config.n_routed_experts >= config.moe_top_k
        assert config.n_shared_experts >= 0

        self.config = config
        self.d_model = config.d_model
        self.n_routed_experts = config.n_routed_experts
        self.n_shared_experts = config.n_shared_experts
        self.top_k = config.moe_top_k

        # Router only chooses among routed experts.
        self.router = nn.Linear(
            config.d_model,
            config.n_routed_experts,
            bias=False,
        )

        self.routed_experts = nn.ModuleList([
            ExpertFFN(config)
            for _ in range(config.n_routed_experts)
        ])

        self.shared_experts = nn.ModuleList([
            ExpertFFN(config)
            for _ in range(config.n_shared_experts)
        ])

    def load_balance_loss(self, router_probs, topk_experts):
        """
        router_probs:
            [num_valid_tokens, n_routed_experts]

        topk_experts:
            [num_valid_tokens, top_k]
        """

        selected_mask = F.one_hot(
            topk_experts,
            num_classes=self.n_routed_experts,
        ).float()
        # [N, top_k, n_routed_experts]

        tokens_per_expert = selected_mask.mean(dim=(0, 1))
        # [n_routed_experts]

        prob_per_expert = router_probs.mean(dim=0)
        # [n_routed_experts]

        aux_loss = self.n_routed_experts * torch.sum(
            tokens_per_expert * prob_per_expert
        )

        return aux_loss

    def forward(self, x, attention_mask=None):
        """
        x:
            [batch_size, seq_len, d_model]

        attention_mask:
            [batch_size, seq_len]

        returns:
            output:   [batch_size, seq_len, d_model]
            aux_loss: scalar
            stats: routing stats
        """

        batch_size, seq_len, d_model = x.shape

        x_flat = x.reshape(-1, d_model)
        # [B*T, D]

        if attention_mask is None:
            valid_token_mask = torch.ones(
                x_flat.shape[0],
                device=x.device,
                dtype=torch.bool,
            )
        else:
            valid_token_mask = attention_mask.reshape(-1).bool()

        x_valid = x_flat[valid_token_mask]
        # [N, D]

        output_flat = torch.zeros_like(x_flat)
        # [B*T, D]

        if x_valid.numel() == 0:
            aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
            stats = {
                "routed_expert_counts": torch.zeros(
                    self.n_routed_experts,
                    device=x.device,
                    dtype=torch.long,
                ),
                "router_probs_mean": torch.zeros(
                    self.n_routed_experts,
                    device=x.device,
                    dtype=x.dtype,
                ),
            }
            return output_flat.view(batch_size, seq_len, d_model), aux_loss, stats

        # -------------------------
        # 1. Shared experts branch
        # -------------------------

        shared_output = torch.zeros_like(x_valid)
        # [N, D]

        for shared_expert in self.shared_experts:
            shared_output = shared_output + shared_expert(x_valid)

        if self.n_shared_experts > 0:
            shared_output = shared_output / self.n_shared_experts

        # -------------------------
        # 2. Routed experts branch
        # -------------------------

        router_logits = self.router(x_valid)
        # [N, n_routed_experts]

        router_probs = torch.softmax(router_logits, dim=-1)
        # [N, n_routed_experts]

        topk_probs, topk_experts = torch.topk(
            router_probs,
            k=self.top_k,
            dim=-1,
        )
        # topk_probs:   [N, top_k]
        # topk_experts: [N, top_k]

        # Normalize only selected expert weights.
        topk_probs = topk_probs / topk_probs.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-9)
        # [N, top_k]

        routed_output = torch.zeros_like(x_valid)
        # [N, D]

        for expert_id, expert in enumerate(self.routed_experts):
            token_idx, expert_slot = (
                topk_experts == expert_id
            ).nonzero(as_tuple=True)

            if token_idx.numel() == 0:
                continue

            expert_input = x_valid[token_idx]
            # [num_tokens_for_expert, D]

            expert_output = expert(expert_input)
            # [num_tokens_for_expert, D]

            weights = topk_probs[token_idx, expert_slot].unsqueeze(-1)
            # [num_tokens_for_expert, 1]

            weighted_output = expert_output * weights

            routed_output.index_add_(
                dim=0,
                index=token_idx,
                source=weighted_output,
            )

        # -------------------------
        # 3. Combine shared + routed
        # -------------------------

        output_valid = shared_output + routed_output
        # [N, D]

        output_flat[valid_token_mask] = output_valid

        output = output_flat.view(batch_size, seq_len, d_model)
        # [B, T, D]

        aux_loss = self.load_balance_loss(
            router_probs=router_probs,
            topk_experts=topk_experts,
        )

        with torch.no_grad():
            routed_expert_counts = torch.bincount(
                topk_experts.reshape(-1),
                minlength=self.n_routed_experts,
            )

            router_probs_mean = router_probs.mean(dim=0)

        stats = {
            "routed_expert_counts": routed_expert_counts,
            "router_probs_mean": router_probs_mean.detach(),
        }

        return output, aux_loss, stats

In [70]:
class DeepSeekMoEDecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = DeepSeekMoEFeedForward(config)

    def forward(self, x, attention_mask=None):
        """
        x: [batch_size, seq_len, d_model]
        """

        x = x + self.attn(
            self.ln_1(x),
            attention_mask=attention_mask,
        )

        moe_out, moe_aux_loss, moe_stats = self.mlp(
            self.ln_2(x),
            attention_mask=attention_mask,
        )

        x = x + moe_out

        if attention_mask is not None:
            x = x * attention_mask[:, :, None].to(x.dtype)

        return x, moe_aux_loss, moe_stats

In [71]:
class DeepSeekMoEGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = TokenEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            DeepSeekMoEDecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:      [B, T]
        attention_mask: [B, T]
        labels:         [B, T]
        """

        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.config.max_seq_len

        x = self.embedding(input_ids)
        # [B, T, D]

        moe_aux_losses = []
        moe_stats_by_layer = []

        for block in self.blocks:
            x, moe_aux_loss, moe_stats = block(
                x,
                attention_mask=attention_mask,
            )

            moe_aux_losses.append(moe_aux_loss)
            moe_stats_by_layer.append(moe_stats)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        # [B, T, vocab_size]

        lm_loss = None
        moe_aux_loss = None
        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            lm_loss = F.cross_entropy(
                shift_logits.reshape(-1, self.config.vocab_size),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

            moe_aux_loss = torch.stack(moe_aux_losses).mean()

            loss = lm_loss + self.config.router_aux_loss_coef * moe_aux_loss

        return {
            "logits": logits,
            "loss": loss,
            "lm_loss": lm_loss,
            "moe_aux_loss": moe_aux_loss,
            "moe_stats_by_layer": moe_stats_by_layer,
        }

In [72]:
torch.manual_seed(42)

batch_size = 3
max_seq_len = 6
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len),
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(attention_mask == 0, -100)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,

    # DeepSeekMoE-style config
    n_routed_experts=8,
    n_shared_experts=1,
    moe_top_k=2,
    moe_hidden_mult=4,
    router_aux_loss_coef=0.01,
)

model = DeepSeekMoEGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(outputs["logits"].shape)

print("\ntotal loss:")
print(outputs["loss"])

print("\nlm loss:")
print(outputs["lm_loss"])

print("\nmoe aux loss:")
print(outputs["moe_aux_loss"])

print("\nDeepSeekMoE routing stats:")
for layer_idx, stats in enumerate(outputs["moe_stats_by_layer"]):
    print(f"\nLayer {layer_idx}")
    print("routed_expert_counts:", stats["routed_expert_counts"])
    print("router_probs_mean:", stats["router_probs_mean"])

outputs["loss"].backward()

print("\nRouter grad exists?")
print(model.blocks[0].mlp.router.weight.grad is not None)

print("\nShared expert grad exists?")
print(model.blocks[0].mlp.shared_experts[0].net[0].weight.grad is not None)

print("\nRouted expert 0 grad exists?")
print(model.blocks[0].mlp.routed_experts[0].net[0].weight.grad is not None)

lengths:
tensor([1, 6, 5])

input_ids:
tensor([[59,  0,  0,  0,  0,  0],
        [ 3, 68, 77, 81, 82, 91],
        [23, 93, 59, 40, 32,  0]])

attention_mask:
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])

labels:
tensor([[  59, -100, -100, -100, -100, -100],
        [   3,   68,   77,   81,   82,   91],
        [  23,   93,   59,   40,   32, -100]])

logits shape:
torch.Size([3, 6, 100])

total loss:
tensor(25.0835, grad_fn=<AddBackward0>)

lm loss:
tensor(25.0727, grad_fn=<NllLossBackward0>)

moe aux loss:
tensor(1.0727, grad_fn=<MeanBackward0>)

DeepSeekMoE routing stats:

Layer 0
routed_expert_counts: tensor([3, 2, 2, 2, 7, 5, 3, 0])
router_probs_mean: tensor([0.1145, 0.1142, 0.1264, 0.1220, 0.1413, 0.1446, 0.1344, 0.1025])

Layer 1
routed_expert_counts: tensor([1, 4, 5, 1, 5, 2, 2, 4])
router_probs_mean: tensor([0.0830, 0.1504, 0.1367, 0.1196, 0.1518, 0.1135, 0.1008, 0.1439])

Router grad exists?
True

Shared expert grad exists?
True

Routed 